In [1]:
import os
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_PLATFORMS"] = "cuda,cpu"

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from numpyro import infer
from numpyro.infer import MCMC, NUTS

IBCD_REPO = Path("/project/bcbg_shared/seong/ibcd_debug/IBCD")
DATA_DIR = Path("/project/bcbg_shared/seong/ibcd_debug/50d")
OUTPUT_DIR = DATA_DIR / "ibcd_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(IBCD_REPO / "src"))
from iv_regression import xi_norm, run_all_IV
from empirical_prior import scale_free_degree_fast, solve_edge_weights_rowwise_directional
from model import matrix_model_spike_horseshoe, compute_lfsr

print("JAX backend devices:", jax.devices())
gpu_devices = [d for d in jax.devices() if d.platform == "gpu"]
assert len(gpu_devices) >= 3, f"Need 3 GPUs; JAX sees {jax.devices()}"

JAX backend devices: [cuda(id=0), cuda(id=1), cuda(id=2)]


In [2]:
# Load Y_matrix.csv and targets.txt
Y_df = pd.read_csv(os.path.join(DATA_DIR, "Y_matrix.csv"))
with open(os.path.join(DATA_DIR, "targets.txt"), "r") as f:
    targets = [line.strip() for line in f if line.strip()]

df = Y_df.copy()
df["target"] = targets

df_control = df[df["target"] == "control"]
df_interventional = df[df["target"] != "control"]
df = pd.concat([df_control, df_interventional], ignore_index=True)

data_csv_path = os.path.join(OUTPUT_DIR, "data.csv")
df.to_csv(data_csv_path, index=False)

print(f"Saved formatted input to: {data_csv_path} (shape: {df.shape})")
print("\nFirst 4 rows (control samples):")
print(df.iloc[:4, list(range(5)) + [-1]].round(2))
print("\nLast 4 rows (interventional samples):")
print(df.iloc[-4:, list(range(5)) + [-1]].round(2))

Saved formatted input to: /project/bcbg_shared/seong/ibcd_debug/50d/ibcd_output/data.csv (shape: (10000, 51))

First 4 rows (control samples):
     V1    V2    V3    V4    V5   target
0 -1.63 -0.73  0.01 -1.63 -0.80  control
1 -0.04 -0.13  0.84  0.64 -0.29  control
2 -0.41  0.59  0.32  0.81  0.91  control
3 -1.77 -0.03 -2.55  0.03  0.45  control

Last 4 rows (interventional samples):
        V1    V2    V3    V4    V5 target
9996  0.98  0.66 -0.00 -0.14  0.17    V50
9997  0.59  1.82  0.44  2.38  0.25    V50
9998  1.45 -1.96  2.53  0.55 -0.76    V50
9999  1.11 -1.11  0.67  1.35  1.14    V50


In [3]:
xi_norm(df, out_dir=OUTPUT_DIR)
run_all_IV(df, out_dir=OUTPUT_DIR)

Rhat_df = pd.read_csv(os.path.join(OUTPUT_DIR, "R.csv"))
colnames = Rhat_df.columns.tolist()
R_hat = Rhat_df.values
xi = pd.read_csv(os.path.join(OUTPUT_DIR, "xi.csv")).values
U_mat = pd.read_csv(os.path.join(OUTPUT_DIR, "U.csv")).values
V_mat = pd.read_csv(os.path.join(OUTPUT_DIR, "V.csv")).values
D = R_hat.shape[0]
print(f"D = {D}")
print("Saved R.csv, U.csv, V.csv, xi.csv")

D = 50
Saved R.csv, U.csv, V.csv, xi.csv



1) Running IV for all targets...: 100%|##########| 50/50 [00:00<00:00, 403.27it/s]


In [4]:
pi0_mat = scale_free_degree_fast(R_hat)
offdiag_mask = ~np.eye(D, dtype=bool)
pi0_i = pi0_mat[offdiag_mask].reshape(D, D - 1).mean(axis=1)
print(
    "Estimated per-node spike weights pi0_i (min / mean / max): "
    f"{pi0_i.min():.4f} / {pi0_i.mean():.4f} / {pi0_i.max():.4f}"
)

pi0_ij, pi_k_ij = solve_edge_weights_rowwise_directional(xi, R_hat, pi0_i)

Estimated per-node spike weights pi0_i (min / mean / max): 0.6136 / 0.9451 / 0.9963


In [5]:
U_lower = jnp.linalg.cholesky(jnp.array(U_mat))
V_lower = jnp.linalg.cholesky(jnp.array(V_mat))

kernel = NUTS(
    matrix_model_spike_horseshoe,
    target_accept_prob=0.7,
    max_tree_depth=10,
    init_strategy=infer.init_to_median(num_samples=50),
)

assert jax.local_device_count() >= 3, (
    f"Need 3 GPU devices for parallel chains; JAX sees {jax.devices()}"
)
chain_method = "parallel"
print("Running 3 MCMC chains in parallel on:", jax.devices())

mcmc = MCMC(
    kernel,
    num_warmup=300,
    num_samples=700,
    num_chains=3,
    chain_method=chain_method,
    progress_bar=True,
)

mcmc.run(
    jax.random.PRNGKey(42),
    obs_data=R_hat,
    pi0_ij=pi0_ij,
    U_lower=U_lower,
    V_lower=V_lower,
    D=D,
)

posterior = np.asarray(
    jax.device_get(mcmc.get_samples(group_by_chain=True)["G"]),
    dtype=np.float32,
)
np.save(os.path.join(OUTPUT_DIR, "G_draws.npy"), posterior)

flat = posterior.reshape(-1, D, D)
posterior_mean = flat.mean(axis=0)
G_est_path = os.path.join(OUTPUT_DIR, "G.csv")
pd.DataFrame(posterior_mean, columns=colnames).to_csv(G_est_path, index=False)

print("Posterior samples shape:", posterior.shape)
print("Saved posterior mean graph to:", G_est_path)

Running 3 MCMC chains in parallel on: [cuda(id=0), cuda(id=1), cuda(id=2)]
Posterior samples shape: (3, 700, 50, 50)
Saved posterior mean graph to: /project/bcbg_shared/seong/ibcd_debug/50d/ibcd_output/G.csv



Compiling.. :   0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1000 [00:00<?, ?it/s]


  0%|          | 0/1000 [00:00<?, ?it/s]


Running chain 0:   0%|          | 0/1000 [00:05<?, ?it/s]

Running chain 1:   0%|          | 0/1000 [00:05<?, ?it/s]


Running chain 0:   5%|5         | 50/1000 [00:23<05:54,  2.68it/s]

Running chain 1:   5%|5         | 50/1000 [00:24<05:58,  2.65it/s]


Running chain 0:  10%|#         | 100/1000 [00:44<05:58,  2.51it/s]

Running chain 1:  10%|#         | 100/1000 [00:44<06:01,  2.49it/s]


Running chain 0:  15%|#5        | 150/1000 [01:02<05:20,  2.65it/s]


Running chain 2:  15%|#5        | 150/1000 [01:03<05:27,  2.60it/s]

Running chain 0:  20%|##        | 200/1000 [01:11<03:58,  3.35it/s]


Running chain 2:  20%|##        | 200/1000 [01:11<04:00,  3.33it/s]

Running chain 0:  25%|##5       | 250/1000 [01:18<03:05,  4.04it/s]


Running chain 2:  25%|##5       | 250/1000 [01:19<03:04,  

In [6]:
def compute_auroc_auprc_from_mats(X_est, X_true, tol=1e-8):
    X_est = np.asarray(X_est, float)
    X_true = np.asarray(X_true, float)
    assert X_est.shape == X_true.shape
    D = X_true.shape[0]
    mask = ~np.eye(D, dtype=bool)
    scores = np.abs(X_est)[mask].ravel()
    y_true = (np.abs(X_true)[mask] > tol).astype(int)
    if y_true.min() == y_true.max():
        return np.nan, np.nan
    y_true = y_true.astype(bool)
    positives = int(y_true.sum())
    negatives = int((~y_true).sum())
    order = np.argsort(-scores, kind="mergesort")
    y_sorted = y_true[order]
    scores_sorted = scores[order]
    distinct = np.r_[scores_sorted[1:] != scores_sorted[:-1], True]
    tp = np.cumsum(y_sorted)[distinct].astype(float)
    fp = np.cumsum(~y_sorted)[distinct].astype(float)
    recall = tp / positives
    precision = tp / (tp + fp)
    auprc = float(np.sum(np.diff(np.r_[0.0, recall]) * precision))
    tpr = np.r_[0.0, recall, 1.0]
    fpr = np.r_[0.0, fp / negatives, 1.0]
    trapz = getattr(np, "trapezoid", np.trapz)
    auroc = float(trapz(tpr, fpr))
    return auroc, auprc


def calc_metrics_r(X, X_true, eps=0.075):
    D = X.shape[1]
    mask = ~np.eye(D, dtype=bool)
    X_off = np.abs(X)[mask].copy()
    X_true_off = np.abs(X_true)[mask].copy()
    X_off[np.abs(X_off) < eps] = 0.0
    X_true_off[np.abs(X_true_off) < eps] = 0.0
    A = X_off > 0
    B = X_true_off > 0
    TP = float(np.sum(A & B))
    FP = float(np.sum(A & (~B)))
    FN = float(np.sum((~A) & B))
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else np.nan
    )
    shd = float(np.sum(A != B))
    return {"F1": f1, "shd": shd}


G_true = pd.read_csv(os.path.join(DATA_DIR, "G_matrix.csv")).values
G_est = pd.read_csv(G_est_path).values
_, auprc = compute_auroc_auprc_from_mats(G_est, G_true, tol=1e-8)
metrics = calc_metrics_r(G_est, G_true, eps=0.1)
print(f"F1    : {metrics['F1']:.6f}")
print(f"SHD   : {metrics['shd']:.0f}")
print(f"AUPRC : {auprc:.6f}")

F1    : 0.973747
SHD   : 11
AUPRC : 0.989929
